In [31]:
import os
import joblib
import re

print(os.path.exists("isolation_forest_model.pkl"))
print(os.path.exists("tfidf_vectorizer.pkl"))
print(os.path.exists("scaler.pkl"))

True
True
True


In [32]:
# Load saved objects
loaded_model = joblib.load("isolation_forest_model.pkl")
loaded_vectorizer = joblib.load("tfidf_vectorizer.pkl")
loaded_scaler = joblib.load("scaler.pkl")


In [33]:
new_logs = [
    "Received block blk_35875081onder: 40051953248 of size 67108864 from /10.251.42.84",
    "deleting block <block_id> file /mnt/hadoop/dfs/data/current/subdir18/<block_id>",
    'Verification succeeded for blk_-4980916519894289629'
"Deleting block blk_1781953582842324563 file /mnt/hadoop/dfs/data/current/subdir5/blk_1781953582842324563",
"  anshdeep"
]


# src/preprocess.py
import re

def preprocess_log(message: str) -> str:
    # Normalize variables
    message = re.sub(r'blk[_-]?\d+', '<block_id>', message)
    message = re.sub(r'\b\d{1,3}(\.\d{1,3}){3}\b', '<ip>', message)
    message = re.sub(r'\b\d{4,5}\b', '<port>', message)
    message = re.sub(r'\b\d{6,}\b', '<large_num>', message)
    message = re.sub(r'\b\d+\b', '<num>', message)

    # Lowercase
    message = message.lower()

    # Clean whitespace
    message = re.sub(r'\s+', ' ', message).strip()

    return message


  

cleaned_messages = [preprocess_log(msg) for msg in new_logs]

X_new = loaded_vectorizer.transform(cleaned_messages)

assert X_new.shape[1] == loaded_model.n_features_in_, \
    f"Feature mismatch: {X_new.shape[1]} vs {loaded_model.n_features_in_}"

X_scaled = loaded_scaler.transform(X_new.toarray())
scores = loaded_model.score_samples(X_scaled)
labels = (loaded_model.predict(X_scaled) == -1).astype(int)

for msg, score, label in zip(cleaned_messages, scores, labels):
    print(f"Message: {msg}")
    print(f"Score: {score:.4f}, Anomaly: {label}")
    print("-" * 40)


Message: received block <block_id>onder: <large_num> of size <large_num> from /<ip>
Score: -0.5416, Anomaly: 1
----------------------------------------
Message: deleting block <block_id> file /mnt/hadoop/dfs/data/current/subdir18/<block_id>
Score: -0.4479, Anomaly: 0
----------------------------------------
Message: verification succeeded for blk_-4980916519894289629deleting block <block_id> file /mnt/hadoop/dfs/data/current/subdir5/<block_id>
Score: -0.4890, Anomaly: 0
----------------------------------------
Message: anshdeep
Score: -0.5264, Anomaly: 1
----------------------------------------
